In [1]:
# 데이터 처리 및 분석
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings

# amenities 전처리
import ast # '['gym']'과 같이 string 형태를 -> ['gym']으로 리스트로 바꿔주는 패키지
import re
from sklearn.preprocessing import MultiLabelBinarizer

# 경로 설정
import os 
import sys

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12,6)

# 시드 설정
np.random.seed(42)

print('-' * 60)
print('라이브러리 로드 완료')
print('한글 폰트 설정 완료!')
print('-'* 60)

------------------------------------------------------------
라이브러리 로드 완료
한글 폰트 설정 완료!
------------------------------------------------------------


In [2]:
os.chdir(os.path.abspath('..'))
print(os.getcwd())

df = pd.read_csv('data/2025_Airbnb_NYC_listings.csv').drop(columns = 'Unnamed: 0')
df.head()

c:\workspace\camp\project\airbnb_price_prediction


,id,source,name,description,neighborhood_overview,host_id,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,36121,city scrape,Lg Rm in Historic Prospect Heights,Cozy space share in the heart of a great neigh...,Full of tree-lined streets and beautiful brown...,62165,Michael,2009-12-11,"New York, NY",I’m an urban planner working for an internatio...,NaN,NaN,NaN,f,Prospect Heights,1.0,3.0,"['email', 'phone', 'work_email']",t,t,Neighborhood highlights,Prospect Heights,Brooklyn,40.673760,-73.966110,Private room in rental unit,Private room,1,1.0,1 shared bath,1.0,1.0,"[""Refrigerator"", ""Dishes and silverware"", ""Wif...",$200.00,90,365,90.0,90.0,365.0,365.0,90.0,365.0,NaN,t,27,57,87,362,2025-03-03,9,0,0,301,0,0,0.0,2010-12-11,2013-05-10,4.88,5.00,4.80,5.00,5.00,5.00,5.00,NaN,f,1,0,1,0,0.05
1,36647,city scrape,"1 Bedroom & your own Bathroom, Elevator Apartment",Private bedroom with your own bathroom in a 2 ...,"Manhattan, SE corner of 2nd Ave/ E. 110th street",157798,Irene,2010-07-04,"New York, NY",NaN,NaN,NaN,100%,f,East Harlem,1.0,1.0,"['email', 'phone']",t,t,Neighborhood highlights,East Harlem,Manhattan,40.792454,-73.940742,Private room in condo,Private room,2,1.0,1 private bath,1.0,1.0,"[""Oven"", ""Blender"", ""Luggage dropoff allowed"",...",$82.00,30,999,30.0,30.0,999.0,999.0,30.0,999.0,NaN,t,0,0,0,204,2025-03-03,102,0,0,143,0,0,0.0,2010-10-04,2023-12-09,4.77,4.82,4.76,4.88,4.90,4.38,4.71,NaN,f,1,0,1,0,0.58
2,38663,city scrape,Luxury Brownstone in Boerum Hill,"Beautiful, large home in great hipster neighbo...","diverse, lively, hip, cool: loaded with restau...",165789,Sarah,2010-07-13,"New York, NY",I am a lawyer and work as an executive at an a...,within a few hours,100%,40%,f,Boerum Hill,1.0,3.0,"['email', 'phone', 'work_email']",t,t,Neighborhood highlights,Boerum Hill,Brooklyn,40.684420,-73.980680,Private room in home,Private room,2,2.5,2.5 baths,5.0,5.0,"[""Portable fans"", ""Oven"", ""Baking sheet"", ""Fir...",$765.00,3,60,3.0,3.0,60.0,60.0,3.0,60.0,NaN,t,30,49,66,326,2025-03-02,43,0,0,267,0,0,0.0,2012-07-09,2023-08-30,4.70,4.83,4.52,4.88,4.88,4.86,4.62,OSE-STRREG-0001784,f,1,0,1,0,0.28
3,38833,city scrape,Spectacular West Harlem Garden Apt,This is a very large and unique space. An inc...,West Harlem is now packed with great restauran...,166532,Matthew,2010-07-14,"New York, NY",I have been a New Yorker for a long time\n and...,within an hour,100%,97%,t,Harlem,1.0,1.0,"['email', 'phone']",t,t,Neighborhood highlights,Harlem,Manhattan,40.818058,-73.946671,Entire home,Entire home/apt,2,1.0,1 bath,1.0,1.0,"[""Fire extinguisher"", ""Clothing storage: close...",$139.00,2,45,2.0,2.0,1125.0,1125.0,2.0,1125.0,NaN,t,7,18,25,25,2025-03-03,241,42,3,25,43,255,35445.0,2010-08-28,2025-02-21,4.85,4.87,4.50,4.96,4.96,4.79,4.82,OSE-STRREG-0000476,

### 1차 전처리

데이터에 필요 없는 칼럼 제거 및 논리에 맞게끔 칼럼 기본 형변환

In [3]:
# 분석에 의미없는 칼럼 제거(단일값 및 스크랩 시점 등)
drop_cols = ['source','calendar_updated','calendar_last_scraped']

# t,f -> True/False로 변환
convert_tf_cols = ['host_is_superhost', 'host_has_profile_pic',
                'host_identity_verified','has_availability',
                'instant_bookable']

# $2,000 -> 2000으로 변환
df['price'] = df['price'].str.replace('$','').str.replace(',','').astype('float')

# 날짜 type으로 변환
df['host_since'] = pd.to_datetime(df['host_since'])
df['first_review'] = pd.to_datetime(df['first_review'])
df['last_review'] = pd.to_datetime(df['last_review'])

# 99% -> 0.99로 변환
df['host_response_rate'] = df['host_response_rate'].str.replace('%','').astype('float') / 100
df['host_acceptance_rate'] = df['host_acceptance_rate'].str.replace('%','').astype('float')  / 100

# 처리 함수
def convert_tf(df, cols):
    for col in cols:
        df[col] = df[col].map({'t':True, 'f':False}).astype('boolean')

    return df

df_cleaned = convert_tf(df, convert_tf_cols)
df_cleaned = df_cleaned.drop(columns = drop_cols)

#### host_verifications

'['email','phone','work_email']'과 같이 리스트가 string형태로 저장

-> 원-핫 인코딩으로 데이터 변환

In [4]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

df_cleaned.loc[df['host_verifications'].isna(),'host_verifications'] = '[]'

# '['email','phone']'과 같이 리스트가 string형태로 저장 -> list로 변환
df_cleaned['host_verifications'] = df_cleaned['host_verifications'].apply(ast.literal_eval)

# Multilabelbinarizier 사용
mlb = MultiLabelBinarizer()

# 리스트 데이터를 0,1로 변환
verifications_encoded = mlb.fit_transform(df_cleaned['host_verifications'])

verifications_encoded

array([[1, 1, 1],
       [1, 1, 0],
       [1, 1, 1],
       ...,
       [1, 1, 1],
       [1, 1, 0],
       [1, 1, 0]], shape=(22308, 3))

In [5]:
# 원-핫 인코딩 저장 정보를 df에 저장
for i, c in enumerate(mlb.classes_):
    col_name = 'host_verifications_' + c
    df_cleaned[col_name] = verifications_encoded[:,i]

df_cleaned = df_cleaned.drop(columns = 'host_verifications')
display(df_cleaned.iloc[:,-3:].head())

,host_verifications_email,host_verifications_phone,host_verifications_work_email
0,1,1,1
1,1,1,0
2,1,1,1
3,1,1,0
4,1,1,0


#### 가격 변수에 대한 log 변환 변수 추가

In [6]:
df_cleaned['log_price'] = np.log(df_cleaned['price'] + 1)
df_cleaned['log_estimated_revenue_l365d'] = np.log(df['estimated_revenue_l365d'] + 1)

#### amenities

전처리 과정(어느 정도 노가다 작업을 진행해서, 간략하게 설명)

1. 영어,숫자,한글만 남김(한자 등 특수문자 제거)

2. MultiLabelBinarizer로 모든 카테고리 원-핫 인코딩 변환

3. 각 column 별로 더 했을 때 1000개 이상의 개수를 기록한 칼럼들에서 가격에 유의미한 지표를 보이거나, 보일 것으로 예상되는 칼럼들 기록
- 에어비엔비 뉴욕 지점에서 했을 때 추천 필터로 나오는 내용들은 필수로 기록
- 사전지식으로 가격과 관계가 있을 것 같은 카테고리 추가
- 판단이 어려운 변수는 보통 제거함. 추가적으로 참고한 것은 log_price와의 평균, pointbiserialr로 상관관계 계산
- OTT는 1,000개 이상은 아니었지만, 보통 다른 서비스와 묶여서 복잡하게 카테고리가 정의되어 있어 단어를 따로 셈

In [7]:
# amenities 구성
df_cleaned['amenities'].str.lower()

0        ["refrigerator", "dishes and silverware", "wif...
1        ["oven", "blender", "luggage dropoff allowed",...
2        ["portable fans", "oven", "baking sheet", "fir...
3        ["fire extinguisher", "clothing storage: close...
4        ["oven", "rice maker", "laundromat nearby", "l...
                               ...                        
22303    ["air conditioning", "carbon monoxide alarm", ...
22304    ["air conditioning", "carbon monoxide alarm", ...
22305    ["air conditioning", "carbon monoxide alarm", ...
22306    ["oven", "gym", "blender", "dedicated workspac...
22307    ["air conditioning", "carbon monoxide alarm", ...
Name: amenities, Length: 22308, dtype: str

In [8]:
df_temp = df_cleaned.copy()
df_temp['amenities_list'] = df_temp['amenities'].apply(ast.literal_eval)

In [9]:
def remove_other_language(amenity):
    cleaned_list = []
    for item in amenity:
        # 영어, 숫자, 공백 제외 모두 제거
        # 제거 후 공백 제거
        cleaned_item = re.sub(r'[^a-zA-Z0-9\s]', '', item).strip().lower()

        if cleaned_item:
            cleaned_list.append(cleaned_item)

    return cleaned_list

df_temp['amenities_list'] = df_temp['amenities_list'].apply(remove_other_language)
df_temp['amenities_list']

0        [refrigerator, dishes and silverware, wifi, ki...
1        [oven, blender, luggage dropoff allowed, dedic...
2        [portable fans, oven, baking sheet, fire extin...
3        [fire extinguisher, clothing storage closet, s...
4        [oven, rice maker, laundromat nearby, luggage ...
                               ...                        
22303    [air conditioning, carbon monoxide alarm, wifi...
22304    [air conditioning, carbon monoxide alarm, wash...
22305    [air conditioning, carbon monoxide alarm, wash...
22306    [oven, gym, blender, dedicated workspace, dini...
22307    [air conditioning, carbon monoxide alarm, wash...
Name: amenities_list, Length: 22308, dtype: object

In [10]:
# 1. MLB 객체 생성
mlb = MultiLabelBinarizer()

# 2. 리스트 데이터를 0과 1의 행렬로 변환
amenities_encoded = mlb.fit_transform(df_temp['amenities_list'])

# 3. 변환된 데이터를 데이터프레임으로 만들고 컬럼명 지정
amenities_df = pd.DataFrame(amenities_encoded, columns=mlb.classes_, index=df_temp.index)
amenities_df = pd.concat([amenities_df, df_cleaned[['price','log_price']]], axis = 1)
amenities_df.head()

,1 bottle of travel sized body wash body soap,1 inch hdtv with amazon prime video apple tv netflix roku,1 inch hdtv with amazon prime video chromecast hulu netflix roku,1 refrigerator,100 inch hdtv with amazon prime video apple tv disney hbo max hulu netflix premium cable,100 inch hdtv with amazon prime video apple tv disney hbo max netflix standard cable,100 inch hdtv with amazon prime video apple tv netflix premium cable standard cable,100 inch hdtv with amazon prime video disney hbo max hulu netflix,100 inch hdtv with apple tv,100 inch hdtv with chromecast,100 inch hdtv with roku,100 inch tv with amazon prime video apple tv netflix roku,100 inch tv with amazon prime video fire tv hulu netflix,110 inch hdtv with amazon prime video apple tv disney netflix,120 inch hdtv with apple tv hbo max netflix amazon prime video,120 inch hdtv with chromecast netflix hulu apple tv hbo max amazon prime video,120 inch hdtv with hbo max,13 inch hdtv with standard cable,14 inch hdtv,14 inch hdtv with amazon prime video fire tv,14 inch tv with roku,140 inch hdtv with hbo max hulu netflix,144 inch tv with dvd player,15 inch hdtv with amazon prime video chromecast disney hbo max netflix,150 inch hdtv with amazon prime video hbo max disney apple tv hulu netflix chromecast,156 inch hdtv with amazon prime video apple tv hbo max netflix,160 inch hdtv with amazon prime video apple tv disney hbo max hulu netflix,17 inch hdtv,18 inch tv with amazon prime video netflix,19 inch hdtv with hulu netflix,19 inch tv,2 burner electric stove,2 burner hot plate electric stove,2 burner induction cooktop we provide pots and pans induction stove,2 in 1 body soap,2 in 1 generic shampoo,20 inch hdtv,20 inch hdtv with amazon prime video netflix,20 inch hdtv with netflix,20 inch tv,20 inch tv with standard cable,24 inch hdtv,24 inch hdtv with amazon prime video apple tv netflix,24 inch hdtv with apple tv,24 inch hdtv with apple tv amazon prime video hulu netflix,24 inch hdtv with chromecast netflix,24 inch hdtv with fire tv amazon prime video,24 inch hdtv with fire tv hulu netflix,24 inch hdtv with hulu netflix,24 inch hdtv with netflix,24 inch hdtv with netflix amazon prime video standard cable,24 inch hdtv with netflix hulu amazon prime video,24 inch hdtv with premium cable standard cable,24 inch hdtv with standard cable,24 inch tv,24 inch tv with amazon prime video disney hulu netflix roku,24 inch tv with amazon prime video netflix roku,24 inch tv with netflix,24 inch tv with netflix amazon prime video,24 inch tv with netflix roku dvd player hulu hbo max disney amazon prime video,25 inch hdtv,25 inch hdtv with amazon prime video hbo max disney,25 inch hdtv with fire tv hbo max netflix amazon prime video apple tv hulu,25 inch hdtv with hbo max hulu netflix amazon prime video,25 inch hdtv with hulu netflix,25 inch hdtv with roku,26 inch hdtv,26 inch hdtv with apple tv,26 inch hdtv with roku,26 inch tv with roku,27 inch hdtv,27 inch hdtv with apple tv,27 inch hdtv with apple tv dvd player,27 inch hdtv with dvd player,27 inch hdtv with netflix amazon prime video,27 inch hdtv with roku,27 inch hdtv with standard cable,27 inch hdtv with standard cable netflix,28 inch hdtv,28 inch hdtv with fire tv roku,28 inch tv with apple tv netflix,3 in 1 body soap,3 in 1 conditioner,3 in 1 organic body wash body soap,3 in 1 organic wash conditioner,3 in 1 organic wash shampoo,3 in one shampoo,3 inch tv with fire tv roku netflix amazon prime video,30 inch hdtv with amazon prime video apple tv disney fire tv hulu netflix,30 inch hdtv with amazon prime video chromecast disney hbo max hulu netflix roku,30 inch hdtv with amazon prime video disney fire tv netflix,30 inch hdtv with amazon prime video fire tv netflix,30 inch hdtv with amazon prime video roku,30 inch hdtv with chromecast,30 inch hdtv with dvd player,30 inch hdtv with hbo max,30 inch hdtv with hbo max premium cable netflix standard cable amazon prime video apple tv,30 inch hdtv with hulu netflix amazon prime video roku chromeca

참고용 기준 1

1. 숙박에 해당 편의시설/서비스가 1000개 이상인 amenities만 확인

In [11]:
amenities_count_df = pd.DataFrame(amenities_df.sum().sort_values(ascending = False), columns = ['values'])
# display(amenities_count_df[amenities_count_df.index.str.contains('apple tv')])
display(amenities_count_df.iloc[2:11].T)
display(amenities_count_df.iloc[101:104].T)

,smoke alarm,wifi,carbon monoxide alarm,kitchen,hot water,essentials,hangers,hair dryer,iron
values,21096.0,20577.0,19386.0,19122.0,17333.0,16548.0,16524.0,15354.0,15268.0


,pack n playtravel crib,coffee maker keurig coffee machine,private backyard fully fenced
values,1132.0,1033.0,989.0


2. pointbiserialr로 이진 범주와 연속형 변수 상관관계 확인

In [12]:
# 참고 기준 2
from scipy.stats import pointbiserialr

x = 'wine glasses'
y = 'log_price'

pointbiserialr(amenities_df[x], amenities_df[y])

SignificanceResult(statistic=np.float64(0.11219981607235625), pvalue=np.float64(2.055726833637635e-63))

참고 기준 3

비슷한 의미를 가지는 단어여도, 가격에 유의미한 영향을 미치는 것만을 선택

ex) view의 경우 정말 여러가지가 있어 개수가 어느 정도 보장되고 가격에 유의미한 영향을 미칠 것 같은 view만 선택

In [13]:
# 개수가 어느 정도 보장되어 있고, 가격 분포의 차이가 있는 애들 뷰만 선택
for idx in amenities_count_df[amenities_count_df.index.str.contains('view')].index:
    print(amenities_df.groupby([idx])['log_price'].mean())
    print(len(amenities_df[amenities_df[idx] == 1]))

city skyline view
0    4.979370
1    5.280838
Name: log_price, dtype: float64
1175
garden view
0    4.992149
1    5.086257
Name: log_price, dtype: float64
735
courtyard view
0    4.994682
1    5.025034
Name: log_price, dtype: float64
417
park view
0    4.995591
1    4.971286
Name: log_price, dtype: float64
314
river view
0    4.992801
1    5.283279
Name: log_price, dtype: float64
188
bay view
0    4.995564
1    4.932254
Name: log_price, dtype: float64
111
beach view
0    4.995204
1    5.012900
Name: log_price, dtype: float64
57
ocean view
0    4.995350
1    4.952024
Name: log_price, dtype: float64
52
harbor view
0    4.994705
1    5.242642
Name: log_price, dtype: float64
49
sea view
0    4.995156
1    5.060197
Name: log_price, dtype: float64
32
pool view
0    4.995159
1    5.083043
Name: log_price, dtype: float64
23
marina view
0    4.994964
1    5.284290
Name: log_price, dtype: float64
22
canal view
0    4.995065
1    5.201024
Name: log_price, dtype: float64
20
lake view
0    4.995276

4. OTT

OTT 서비스는 보통 hdtv with apple tv 같이 다른 서비스와 결합되어 적혀있으므로, 주로 적히는 ott의 단어의 개수를 세서 추출

In [14]:
# 단순히 단어만 세었을 때, OTT가 상위권에 주로 포진해있는 것을 알 수 있음

from collections import Counter

all_words = []
for amenities in amenities_df.columns:
    all_words.extend(amenities.split())

word_counts = Counter(all_words)

sorted_counts = sorted(word_counts.items(), key = lambda x: x[1], reverse = True)
sorted_word_dict = dict(sorted_counts)
display(sorted_word_dict)

{'with': 1969,
 'hdtv': 1552,
 'inch': 1407,
 'netflix': 1402,
 'tv': 1359,
 'amazon': 1133,
 'prime': 1122,
 'video': 1122,
 'hulu': 863,
 'hbo': 699,
 'max': 699,
 'cable': 688,
 'disney': 618,
 'soap': 605,
 'body': 603,
 'apple': 599,
 'wifi': 599,
 'mbps': 594,
 'roku': 563,
 'fast': 553,
 'conditioner': 463,
 'shampoo': 454,
 'standard': 445,
 'fire': 426,
 'oven': 351,
 'stainless': 297,
 'steel': 297,
 'stove': 276,
 'refrigerator': 264,
 'sound': 260,
 'premium': 259,
 'system': 253,
 'available': 247,
 'and': 238,
 'chromecast': 229,
 '55': 214,
 '50': 185,
 'bluetooth': 181,
 'gas': 160,
 '65': 149,
 'coffee': 119,
 '32': 106,
 'housekeeping': 104,
 '42': 98,
 'single': 92,
 'electric': 90,
 '40': 86,
 'at': 85,
 'to': 83,
 'dove': 81,
 'maker': 81,
 'player': 79,
 'in': 76,
 'aux': 75,
 'hours': 74,
 'extra': 72,
 'dvd': 71,
 'a': 71,
 'free': 71,
 'cost': 71,
 '60': 67,
 'pool': 67,
 'day': 65,
 'from': 59,
 'shared': 59,
 '43': 58,
 'on': 53,
 'brand': 52,
 'the': 51,
 'a

In [15]:
amenities_count_df = pd.DataFrame(amenities_df.sum().sort_values(ascending = False), columns = ['values'])
display(amenities_count_df[amenities_count_df.index.str.contains('netflix')])
# amenities_count_df.iloc[0:50]
amenities_count_df[amenities_count_df.index.str.contains('netflix')].sum()

,values
hdtv with netflix,77.0
tv with netflix,42.0
55 inch hdtv with netflix,41.0
32 inch hdtv with fire tv netflix,17.0
50 inch hdtv with netflix,17.0
...,...
120 inch hdtv with chromecast netflix hulu apple tv hbo max amazon prime video,1.0
120 inch hdtv with apple tv hbo max netflix amazon prime video,1.0
110 inch hdtv with amazon prime video apple tv disney netflix,1.0
100 inch tv with amazon prime video fire tv hulu netflix,1.0


values    2052.0
dtype: float64

- 앞서 언급한 기준을 참고해서, 다음과 같이 카테고리 선택

1. 에어비엔비 추천 필터 요소

```python
# 셀프체크인 유무
has_selfcheckin = ['self checkin']

# 욕실(tub 찾기)
tub = ['tub']

# 무료 주차 공간(parking만 필터링 후 free 포함한 단어 찾기)
free_parking = ['free street parking','free parking on premises']

# 와이파이(wifi 단어 포함 여부)
wifi = ['wifi']

# TV(tv 포함 여부)
tv = ['tv']

# 세탁기 (세탁기 포함. paid 제외)
washer = ['washer']

# 주방('kitchen' 띄어쓰기로 추출해보기)
kitchen = ['kitchen']

# 건조기 (dryer. hair제외)
dryer = ['dryer']

# 에어컨
air_conditioning = ['air conditioning']

# 난방(heating 포함 여부)
heating = ['heating']

# 업무 전용 공간
dedicated_workspace = ['dedicated workspace']

# 헤어드라이어
hair_dryer = ['hair_dryer']

# 다리미
iron = ['iron']

# 수영장 (' pool' 포함한 단어로, 'whirl', 'worl' 제외)
pool = ['pool']

# 헬스장
gym = ['gym']

# 바비큐 그릴
bbq_grill = ['bbq grill']

# 조식
breakfast = ['breakfast']

# 벽난로 ('fireplace' 포함 여부)
fireplace = ['fireplace']

# 흡연 가능 여부
smoking_allowed	= ['smoking allowed']

# 안전 요소
safety = ['smoke alarm', 'carbon monoxide alarm']

# 아기용 침대
crib = ['crib']

# 반려동물 출입 여부
pets = ['pets allowed']
```

2. 그 외 사전지식으로 관련있는 카테고리를 사전지식으로 그룹화 함

```python
# 비싼 가전 제품 개수
appliances = ['microwave', 'oven', 'refrigerator', 'freezer', 'dishwasher', 'stove', 'coffee maker', 'hot water kettle','toaster','blender']

# 인테리어 요소/고급 숙소 추측
wine_glasses = ['wine glasses']

# 야외 공간 유무
outdoor = ['outdoor furniture', 'outdoor dining area', 'private patio or balcony','patio or balcony']

# 수하물 보관 허용
has_luggage = ['luggage dropoff allowed']

# 장기 숙박 유무
has_long_term = ['long term stays allowed']

# 독립된 체크인 유무
has_private = ['private entrance']

# 무료 주차 관련
free_parking = ['free street parking','free parking on premises']

# CCTV
cctv = ['exterior security cameras']

# 엘리베이터 유무
elevator = ['elevator']

# 개인실 일 경우 활용해야 할 듯
lock_bedroom = ['lock on bedroom door']

# view
# city skyline view
# garden view
# courtyard view
# river view

# OTT
# netflix, roku, disney, ...
ott_list = ['netflix', 'hulu', 'disney', 'hbo', 'amazon']
```

In [16]:
# 문자열 정규표현식으로 영어,숫자,공백만 추출
df_cleaned['amenities'] = df_cleaned['amenities'].str.lower().str.replace(r'[^a-zA-Z0-9\s\"\,]', '', regex = True)

1. has_amenities 변수 생성

포함하면 1, 아니면 0

In [17]:
def tf_amenities(df, count_df, x, exclude = None):
    mask1 = count_df.index.str.contains(x)

    if exclude:
        mask2 = ~ count_df.index.str.contains(exclude)
        tmp = count_df[mask1 & mask2].index
    else:
        tmp = count_df[mask1].index
    
    if len(tmp) == 0:
        return 0
    return df[tmp].max(axis = 1)

# 그냥 바로 함수 돌려도 되는 변수
clean_cols = ['self checkin', 'wifi', 'kitchen', 'dryer', 'air conditioning', 'heating', 'dedicated workspace', 'hair dryer', 'iron',
            'gym', 'bbq grill', 'breakfast', 'fireplace', 'smoking allowed', 'pets allowed', 'wine glasses', 'crib', 'city skyline view',
            'river view', 'luggage dropoff allowed', 'long term stays allowed', 'private entrance', 
            'exterior security cameras', 'elevator', 'lock on bedroom door']

for col in clean_cols:
    col_name = 'has_' + col.replace(' ', '_')
    df_temp[col_name] = tf_amenities(amenities_df, amenities_count_df, col)

df_temp.iloc[:,-len(clean_cols):].head()

,has_self_checkin,has_wifi,has_kitchen,has_dryer,has_air_conditioning,has_heating,has_dedicated_workspace,has_hair_dryer,has_iron,has_gym,has_bbq_grill,has_breakfast,has_fireplace,has_smoking_allowed,has_pets_allowed,has_wine_glasses,has_crib,has_city_skyline_view,has_river_view,has_luggage_dropoff_allowed,has_long_term_stays_allowed,has_private_entrance,has_exterior_security_cameras,has_elevator,has_lock_on_bedroom_door
0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0
1,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0
2,1,1,1,1,1,1,1,1,1,0,1,0,1,0,0,1,1,0,0,1,1,1,0,0,0
3,0,1,1,1,0,1,1,1,1,0,0,0,0,0,0,1,0,0,0,1,1,1,0,0,0
4,1,1,1,1,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0


In [18]:
# 제외해야 할 변수가 있는 경우는 따로 돌림
df_temp['has_tub'] = tf_amenities(amenities_df, amenities_count_df, 'tub', 'shared')
df_temp['has_washer'] = tf_amenities(amenities_df, amenities_count_df, 'washer', 'paid|dish')
df_temp['has_dryer'] = tf_amenities(amenities_df, amenities_count_df, 'dryer', 'paid|hair')
df_temp['has_pool'] = tf_amenities(amenities_df, amenities_count_df, 'pool', 'whirl|worl|world|refrigerator|oven|stove|table')

# 여러 변수를 하나의 변수로 합치는 경우
df_temp['has_alarm'] = tf_amenities(amenities_df, amenities_count_df, 'smoke alarm|carbon monoxide alarm')
df_temp['has_garden_courtyard_view'] = tf_amenities(amenities_df, amenities_count_df, 'garden view|courtyard view')
df_temp['has_outdoor'] = tf_amenities(amenities_df, amenities_count_df, 'outdoor furniture|outdoor dining area|private patio or balcony|patio or balcony')

x = 'free parking'
tmp = amenities_count_df[amenities_count_df.index.str.contains(x)].index
df_temp['has_free_parking'] = amenities_df[['free street parking'] + list(tmp)].max(axis = 1)

In [19]:
# amenities 전체 개수
df_temp['amenities_count'] = df_temp['amenities_list'].apply(lambda x: len(x))

# 비싼 가전 제품 개수
appliances_expensive = ['microwave', 'oven', 'refrigerator', 'freezer', 'dishwasher', 'stove', 'coffee maker']
df_temp['count_appliances_expensive'] = amenities_df[appliances_expensive].sum(axis = 1)

# OTT 서비스 유무
df_temp['has_ott'] = tf_amenities(amenities_df, amenities_count_df, 'netflix|amazon prime|disney|apple tv|hbo max|hulu|roku')

In [20]:
df_temp.iloc[:,-41:].columns

Index(['host_verifications_email', 'host_verifications_phone',
       'host_verifications_work_email', 'log_price',
       'log_estimated_revenue_l365d', 'amenities_list', 'has_self_checkin',
       'has_wifi', 'has_kitchen', 'has_dryer', 'has_air_conditioning',
       'has_heating', 'has_dedicated_workspace', 'has_hair_dryer', 'has_iron',
       'has_gym', 'has_bbq_grill', 'has_breakfast', 'has_fireplace',
       'has_smoking_allowed', 'has_pets_allowed', 'has_wine_glasses',
       'has_crib', 'has_city_skyline_view', 'has_river_view',
       'has_luggage_dropoff_allowed', 'has_long_term_stays_allowed',
       'has_private_entrance', 'has_exterior_security_cameras', 'has_elevator',
       'has_lock_on_bedroom_door', 'has_tub', 'has_washer', 'has_pool',
       'has_alarm', 'has_garden_courtyard_view', 'has_outdoor',
       'has_free_parking', 'amenities_count', 'count_appliances_expensive',
       'has_ott'],
      dtype='str')

#### review 관련

review의 경우 number_of_reviews가 0인 경우, 나머지 reviews들은 결측치로 표기되어있음.

-> 0으로 대체해 '리뷰 없음'의 의미를 담게끔 결측치 대체

In [21]:
# 리뷰 관련 컬럼 리스트로
review_columns = [
    "number_of_reviews",
    "reviews_per_month",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",
    "first_review",
    "last_review",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
]

df_reviews = df_cleaned[review_columns]

display(df_reviews.head())

,number_of_reviews,reviews_per_month,number_of_reviews_ltm,number_of_reviews_l30d,number_of_reviews_ly,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value
0,9,0.05,0,0,0,2010-12-11,2013-05-10,4.88,5.00,4.80,5.00,5.00,5.00,5.00
1,102,0.58,0,0,0,2010-10-04,2023-12-09,4.77,4.82,4.76,4.88,4.90,4.38,4.71
2,43,0.28,0,0,0,2012-07-09,2023-08-30,4.70,4.83,4.52,4.88,4.88,4.86,4.62
3,241,1.36,42,3,43,2010-08-28,2025-02-21,4.85,4.87,4.50,4.96,4.96,4.79,4.82
4,274,1.54,12,0,12,2010-08-02,2025-01-03,4.82,4.83,4.61,4.94,4.88,4.85,4.78


In [22]:
# 리뷰 컬럼별 결측치 개수
print(df_reviews.isnull().sum())

# 결측치 비율로
reviews_missing_ratio = (df_reviews.isnull().sum() / len(df_reviews)) * 100
print(reviews_missing_ratio.round(2))

number_of_reviews                 0
reviews_per_month              6798
number_of_reviews_ltm             0
number_of_reviews_l30d            0
number_of_reviews_ly              0
first_review                   6798
last_review                    6798
review_scores_rating           6798
review_scores_accuracy         6798
review_scores_cleanliness      6798
review_scores_checkin          6798
review_scores_communication    6798
review_scores_location         6799
review_scores_value            6798
dtype: int64
number_of_reviews               0.00
reviews_per_month              30.47
number_of_reviews_ltm           0.00
number_of_reviews_l30d          0.00
number_of_reviews_ly            0.00
first_review                   30.47
last_review                    30.47
review_scores_rating           30.47
review_scores_accuracy         30.47
review_scores_cleanliness      30.47
review_scores_checkin          30.47
review_scores_communication    30.47
review_scores_location         30.48
re

In [23]:
# 결측치 원인
no_reviews_count = (df_reviews["number_of_reviews"] == 0).sum()
print(f"리뷰가 없는 숙소의 수: {no_reviews_count}개")

리뷰가 없는 숙소의 수: 6798개


In [24]:
# df_reviews_cleaned 에 리뷰 컬럼 모음 복사본 저장
df_reviews_cleaned = df_temp.copy()

# 월별 리뷰 수 결측치 0(리뷰 없음)으로 채우기
df_reviews_cleaned[["reviews_per_month"]] = df_reviews_cleaned[["reviews_per_month"]].fillna(0)

# 평점 관련 컬럼 필터링
score_columns = df_reviews_cleaned.filter(like="review_scores").columns
print(score_columns)

# 각 평점 컬럼 결측치 0(리뷰 없음)으로 채우기
df_reviews_cleaned[score_columns] = df_reviews_cleaned[score_columns].fillna(0)

# df_reviews_cleaned
print(df_reviews_cleaned[score_columns].isna().sum())

Index(['review_scores_rating', 'review_scores_accuracy',
       'review_scores_cleanliness', 'review_scores_checkin',
       'review_scores_communication', 'review_scores_location',
       'review_scores_value'],
      dtype='str')
review_scores_rating           0
review_scores_accuracy         0
review_scores_cleanliness      0
review_scores_checkin          0
review_scores_communication    0
review_scores_location         0
review_scores_value            0
dtype: int64


In [25]:
df_final = df_reviews_cleaned.copy()
df_final.to_csv('data/df_cleaned.csv', index = False)